In [288]:
! pip install "granite-tsfm[notebooks]==0.3.1"
! pip install gdown


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [289]:
import math
import os

import numpy as np
import pandas as pd
import torch
import evaluate
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import Subset
from transformers import EarlyStoppingCallback, Trainer, TrainingArguments, set_seed

import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score

from tsfm_public.models.tspulse import TSPulseForReconstruction
from tsfm_public.toolkit.dataset import PretrainDFDataset
from tsfm_public import get_datasets
from tsfm_public.toolkit.ad_helpers import AnomalyScoreMethods
from tsfm_public.toolkit.time_series_anomaly_detection_pipeline import TimeSeriesAnomalyDetectionPipeline
from tsfm_public.toolkit.time_series_preprocessor import TimeSeriesPreprocessor, prepare_data_splits

## Load data from carOBD

In [290]:
path = "/Users/darenpalmer/Desktop/UCL/CS/fyp.nosync/data/carOBD/obdiidata" 
time_col = 'ENGINE_RUN_TINE ()'

df_list = []
for file in os.listdir(path):
  if file.endswith('.csv'):
    df = pd.read_csv(f'{path}/{file}', index_col=False)
    df['filename'] = file
    df_list.append(df)

print(f'{len(df_list)} files loaded out of {len([f for f in os.listdir(path) if f.endswith(".csv")])}')


129 files loaded out of 129


## Remove zero-variance columns

In [291]:
def remove_zero_variance_columns(df: pd.DataFrame) -> pd.DataFrame:
  """
  Compute std of each std-computable column (numeric only)
  """
  std_df = df.std(numeric_only=True)

  zero_variance_cols = std_df[std_df == 0].index.tolist()
  print(f'{len(zero_variance_cols)} columns with zero variance')

  if len(zero_variance_cols) > 0:
    df = df.drop(columns=zero_variance_cols)

  return df

## Handle missing Timestamps and duplicates

In [292]:
def mean_fill_missing_timestamps_and_remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
  """
  Remove duplicate timestamps by averaging all numeric columns for each unique timestamp.
  This preserves the overall statistics while removing duplicate entries.
  
  Note: The time column itself is not averaged (it becomes the group key).
  Only numeric columns are averaged when multiple rows share the same timestamp.
  """
  df = df.groupby(time_col, as_index=False).mean(numeric_only=True)
  return df

## Inject Anomalies

In [293]:
from typing import Tuple, Dict, Optional, List

def create_realistic_fault_data(
    normal_data: np.ndarray, 
    fault_percentage: float = 0.2,
    random_state: Optional[int] = None,
    fault_types: Optional[List[str]] = None
) -> Tuple[np.ndarray, np.ndarray, Dict]:
    if fault_types is None:
        fault_types = ['coolant_bias', 'coolant_drift', 'coolant_stuck', 'rpm_bias', 'multi_sensor']
    
    if random_state is not None:
        np.random.seed(random_state)
    
    n_samples = len(normal_data)
    n_faults = int(n_samples * fault_percentage)
    
    fault_labels = np.zeros(n_samples, dtype=int)
    fault_indices = np.random.choice(n_samples, n_faults, replace=False)
    fault_labels[fault_indices] = 1
    fault_features = normal_data.copy()
    
    fault_type_list = [None] * n_samples
    original_values = [[] for _ in range(n_samples)]
    modified_values = [[] for _ in range(n_samples)]
    
    for idx in fault_indices:
        fault_type = np.random.choice(fault_types)
        fault_type_list[idx] = fault_type
        
        if fault_type == 'coolant_bias':
            bias_amount = np.random.uniform(40, 80) * np.random.choice([-1, 1])
            original_val = fault_features[idx, 0]
            fault_features[idx, 0] += bias_amount
            original_values[idx].append(original_val)
            modified_values[idx].append(fault_features[idx, 0])
        
        elif fault_type == 'coolant_drift':
            drift_factor = np.random.uniform(0.6, 1.4)
            original_val = fault_features[idx, 0]
            fault_features[idx, 0] *= drift_factor
            original_values[idx].append(original_val)
            modified_values[idx].append(fault_features[idx, 0])
            
            if fault_features.shape[1] > 1:
                rpm_factor = np.random.uniform(0.7, 1.3)
                original_rpm = fault_features[idx, 1]
                fault_features[idx, 1] *= rpm_factor
                original_values[idx].append(original_rpm)
                modified_values[idx].append(fault_features[idx, 1])
        
        elif fault_type == 'coolant_stuck':
            stuck_temp = np.random.uniform(10, 150)
            original_val = fault_features[idx, 0]
            fault_features[idx, 0] = stuck_temp
            original_values[idx].append(original_val)
            modified_values[idx].append(stuck_temp)
        
        elif fault_type == 'rpm_bias':
            rpm_bias = np.random.uniform(300, 1000) * np.random.choice([-1, 1])
            if fault_features.shape[1] > 1:
                original_rpm = fault_features[idx, 1]
                fault_features[idx, 1] += rpm_bias
                original_values[idx].append(original_rpm)
                modified_values[idx].append(fault_features[idx, 1])
        
        elif fault_type == 'multi_sensor':
            degradation_factor = np.random.uniform(0.5, 1.5)
            original_vals_copy = fault_features[idx].copy()
            fault_features[idx] *= degradation_factor
            noise_level = np.random.uniform(0.15, 0.3)
            fault_features[idx] += np.random.normal(0, noise_level, fault_features.shape[1])
            
            for col_idx in range(fault_features.shape[1]):
                original_values[idx].append(original_vals_copy[col_idx])
                modified_values[idx].append(fault_features[idx, col_idx])
    
    fault_info = {
        'fault_types': fault_type_list,
        'original_values': original_values,
        'modified_values': modified_values,
        'n_faults': n_faults,
        'fault_percentage': fault_percentage
    }
    
    return fault_features, fault_labels, fault_info

## Standardization

In [294]:
from sklearn.preprocessing import StandardScaler

def scale_data(normal_data: np.ndarray, fault_data: np.ndarray = None) -> Tuple[np.ndarray, StandardScaler, np.ndarray]:
    scaler = StandardScaler()
    normal_scaled = scaler.fit_transform(normal_data)
    
    if fault_data is not None:
        fault_scaled = scaler.transform(fault_data)
        return normal_scaled, scaler, fault_scaled
    
    return normal_scaled, scaler, None

## Process Data

In [295]:
target_columns = [
    'COOLANT_TEMPERATURE ()',
    'ENGINE_RPM ()',
    'VEHICLE_SPEED ()',
    'THROTTLE ()',
    'ENGINE_LOAD ()',
    'INTAKE_MANIFOLD_PRESSURE ()',
]

all_file_data = skipped_files = []

def process_file(df, target_columns, fault_percentage=0.3, random_state=42):
    """Process a single file and return data + masks with VALIDATION"""
    df_clean = remove_zero_variance_columns(
        mean_fill_missing_timestamps_and_remove_duplicates(df)
    )
    
    missing_cols = [col for col in target_columns if col not in df_clean.columns]
    if missing_cols:
        print(f"⚠️  File missing columns: {missing_cols}")
        return None 

    feature_cols = [col for col in df_clean.columns if col != time_col]
    normal_features = df_clean[feature_cols].values
    
    fault_features, fault_labels, fault_info = create_realistic_fault_data(
        normal_features, 
        fault_percentage=fault_percentage,
        random_state=random_state
    )
    
    fault_df = df_clean.copy()
    fault_df[feature_cols] = fault_features
    fault_df['anomaly_label'] = fault_labels
    
    # Extract ONLY target columns in the correct order
    try:
        data = fault_df[target_columns].values.astype(np.float32)
    except KeyError as e:
        print(f"⚠️  Error extracting columns: {e}")
        return None
    
    expected_shape = (len(data), len(target_columns))
    if data.shape != expected_shape:
        print(f"⚠️  Shape mismatch: expected {expected_shape}, got {data.shape}")
        return None
    
    observed_mask = (1 - fault_df['anomaly_label'].values).astype(bool)
    observed_mask = np.expand_dims(observed_mask, 1).repeat(len(target_columns), axis=1)
    
    return data, observed_mask, fault_info


print("Processing all files...")
all_file_data = []

for idx, df in enumerate(df_list):
    result = process_file(df, target_columns, fault_percentage=0.3, random_state=42+idx)
    
    if result is None:
        skipped_files.append(idx)
        print(f"❌ Skipped file {idx}")
        continue
    
    data, mask, info = result
    
    # Validate dimensions
    assert data.shape[1] == len(target_columns), f"File {idx}: Expected {len(target_columns)} channels, got {data.shape[1]}"
    assert mask.shape == data.shape, f"File {idx}: Mask shape {mask.shape} doesn't match data shape {data.shape}"
    
    all_file_data.append({
        'file_idx': idx,
        'data': data,
        'mask': mask,
        'info': info,
        'n_samples': len(data)
    })
    print(f"✓ File {idx}: {len(data)} samples, shape {data.shape}")

print(f"\nTotal files: {len(all_file_data)}")
print(f"Total samples: {sum(f['n_samples'] for f in all_file_data)}")    

Processing all files...
5 columns with zero variance
✓ File 0: 617 samples, shape (617, 6)
4 columns with zero variance
✓ File 1: 722 samples, shape (722, 6)
4 columns with zero variance
✓ File 2: 512 samples, shape (512, 6)
3 columns with zero variance
✓ File 3: 1111 samples, shape (1111, 6)
5 columns with zero variance
⚠️  File missing columns: ['VEHICLE_SPEED ()']
❌ Skipped file 4
5 columns with zero variance
⚠️  File missing columns: ['VEHICLE_SPEED ()']
❌ Skipped file 5
4 columns with zero variance
✓ File 6: 619 samples, shape (619, 6)
4 columns with zero variance
✓ File 7: 626 samples, shape (626, 6)
4 columns with zero variance
✓ File 8: 957 samples, shape (957, 6)
4 columns with zero variance
✓ File 9: 349 samples, shape (349, 6)
4 columns with zero variance
✓ File 10: 504 samples, shape (504, 6)
6 columns with zero variance
⚠️  File missing columns: ['VEHICLE_SPEED ()']
❌ Skipped file 11
7 columns with zero variance
⚠️  File missing columns: ['VEHICLE_SPEED ()']
❌ Skipped file

## Metrics

In [296]:
from evaluate import load

metric = load("f1")

In [297]:
## Visualization Function

def plot_anomaly_detection(
    results: pd.DataFrame,
    timestamp_column: str,
    target_columns: list,
    ground_truth_labels: np.ndarray = None,
    threshold: float = 0.5,
    figsize: tuple = (15, 12)
):
    anomaly_scores = results['anomaly_score'].values
    predicted_anomalies = anomaly_scores >= threshold
    
    if ground_truth_labels is not None:
        injected_anomalies = ground_truth_labels == 1
        tp = (injected_anomalies & predicted_anomalies).sum()
        fp = (~injected_anomalies & predicted_anomalies).sum()
        fn = (injected_anomalies & ~predicted_anomalies).sum()
        tn = (~injected_anomalies & ~predicted_anomalies).sum()
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    n_plots = len(target_columns) + 2 if ground_truth_labels is not None else len(target_columns) + 1
    fig, axes = plt.subplots(n_plots, 1, figsize=figsize)
    fig.suptitle('Anomaly Detection: Injected vs Predicted', fontsize=16, fontweight='bold')
    
    if n_plots == 1:
        axes = [axes]
    
    for idx, col in enumerate(target_columns):
        if col not in results.columns:
            continue
        ax = axes[idx]
        ax.plot(results[timestamp_column], results[col], label=col, alpha=0.6, linewidth=1, color='gray')
        
        if ground_truth_labels is not None:
            if injected_anomalies.sum() > 0:
                ax.scatter(
                    results[timestamp_column][injected_anomalies],
                    results[col][injected_anomalies],
                    color='red', s=40, marker='x', label='Injected', zorder=5, alpha=0.8, linewidths=2
                )
            if predicted_anomalies.sum() > 0:
                ax.scatter(
                    results[timestamp_column][predicted_anomalies],
                    results[col][predicted_anomalies],
                    color='orange', s=30, marker='o', label='Predicted', zorder=4, alpha=0.6, edgecolors='darkorange'
                )
            if tp > 0:
                tp_mask = injected_anomalies & predicted_anomalies
                ax.scatter(
                    results[timestamp_column][tp_mask],
                    results[col][tp_mask],
                    color='green', s=50, marker='*', label='True Positive', zorder=6, alpha=0.9
                )
        else:
            if predicted_anomalies.sum() > 0:
                ax.scatter(
                    results[timestamp_column][predicted_anomalies],
                    results[col][predicted_anomalies],
                    color='red', s=30, label='Anomaly', zorder=5, alpha=0.8
                )
        
        ax.set_ylabel(col, fontsize=10)
        ax.legend(loc='upper right', fontsize=8)
        ax.grid(True, alpha=0.3)
    
    ax = axes[-2] if ground_truth_labels is not None else axes[-1]
    ax.plot(results[timestamp_column], anomaly_scores, label='Anomaly Score', color='purple', linewidth=1)
    ax.axhline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold ({threshold})')
    ax.set_xlabel('Time', fontsize=10)
    ax.set_ylabel('Anomaly Score', fontsize=10)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)
    
    if ground_truth_labels is not None:
        ax = axes[-1]
        comparison = np.zeros(len(results))
        comparison[injected_anomalies & predicted_anomalies] = 1
        comparison[injected_anomalies & ~predicted_anomalies] = 2
        comparison[~injected_anomalies & predicted_anomalies] = 3
        
        colors_map = {0: 'gray', 1: 'green', 2: 'red', 3: 'orange'}
        labels_map = {0: 'Normal', 1: 'TP', 2: 'FN', 3: 'FP'}
        
        for val in [1, 2, 3]:
            mask = comparison == val
            if mask.sum() > 0:
                ax.scatter(
                    results[timestamp_column][mask],
                    comparison[mask],
                    color=colors_map[val], s=30, label=labels_map[val], alpha=0.7
                )
        
        ax.set_xlabel('Time', fontsize=10)
        ax.set_ylabel('Classification', fontsize=10)
        ax.set_yticks([0, 1, 2, 3])
        ax.set_yticklabels(['Normal', 'TP', 'FN', 'FP'])
        ax.legend(loc='upper right', fontsize=8)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n{'='*60}")
    print("Anomaly Detection Statistics")
    print(f"{'='*60}")
    print(f"Total points: {len(results):,}")
    print(f"Predicted anomalies (≥{threshold}): {predicted_anomalies.sum():,} ({predicted_anomalies.sum()/len(results)*100:.2f}%)")
    
    if ground_truth_labels is not None:
        print(f"\nGround Truth:")
        print(f"  Injected anomalies: {injected_anomalies.sum():,} ({injected_anomalies.sum()/len(results)*100:.2f}%)")
        print(f"\nConfusion Matrix:")
        print(f"  True Positives (TP):  {tp:,}")
        print(f"  False Positives (FP): {fp:,}")
        print(f"  False Negatives (FN): {fn:,}")
        print(f"  True Negatives (TN):  {tn:,}")
        print(f"\nMetrics:")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall:    {recall:.4f}")
        print(f"  F1-Score:  {f1:.4f}")
    
    print(f"\nAnomaly Score Statistics:")
    print(f"  Min: {anomaly_scores.min():.4f}, Max: {anomaly_scores.max():.4f}")
    print(f"  Mean: {anomaly_scores.mean():.4f}, Std: {anomaly_scores.std():.4f}")
    print(f"  Percentiles - 50th: {np.percentile(anomaly_scores, 50):.4f}, 95th: {np.percentile(anomaly_scores, 95):.4f}, 99th: {np.percentile(anomaly_scores, 99):.4f}")



In [298]:
def create_windows_from_files(file_indices, all_file_data, context_length, stride):
    """Create windows from multiple files"""
    all_windows = []
    all_masks = []
    file_sources = []  # Track which file each window came from
    
    for file_idx in file_indices:
        file_data = all_file_data[file_idx]
        data = file_data['data']
        mask = file_data['mask']
        
        # Create windows from this file
        for start_idx in range(0, len(data) - context_length + 1, stride):
            end_idx = start_idx + context_length
            all_windows.append(data[start_idx:end_idx])
            all_masks.append(mask[start_idx:end_idx])
            file_sources.append(file_idx)
        
        # Handle last window with padding
        if len(data) >= context_length:
            start_idx = len(data) - context_length
            if start_idx not in range(0, len(data) - context_length + 1, stride):
                window_data = data[start_idx:]
                window_mask = mask[start_idx:]
                
                if len(window_data) < context_length:
                    pad_len = context_length - len(window_data)
                    window_data = np.vstack([window_data, np.zeros((pad_len, data.shape[1]))])
                    window_mask = np.vstack([window_mask, np.ones((pad_len, data.shape[1]), dtype=bool)])
                
                all_windows.append(window_data)
                all_masks.append(window_mask)
                file_sources.append(file_idx)
    
    return np.array(all_windows), np.array(all_masks), file_sources


## Fine-tune Model

In [299]:
from tsfm_public.models.tspulse import TSPulseForReconstruction
from tsfm_public.models.tspulse.utils.helpers import PatchMaskingDatasetWrapper
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

set_seed(42)

target_columns = [
    'COOLANT_TEMPERATURE ()',
    'ENGINE_RPM ()',
    'VEHICLE_SPEED ()',
    'THROTTLE ()',
    'ENGINE_LOAD ()',
    'INTAKE_MANIFOLD_PRESSURE ()',
]

context_length = 512
prediction_length = 16

column_specifiers = {
    "timestamp_column": time_col,
    "target_columns": target_columns,
    "id_columns": [],
}

model = TSPulseForReconstruction.from_pretrained(
    "ibm-granite/granite-timeseries-tspulse-r1",
    num_input_channels=len(target_columns),
    revision='main',
    decoder_mode="common_channel", # TODO: test with more samples and mix_channel
    scaling="revin",
    mask_type="user"
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model.to(device)

# freeze backbone, and only train decoder and head since we have a small dataset

for param in model.backbone.parameters():
    param.requires_grad = False

for module in model.modules():
    if isinstance(module, (torch.nn.BatchNorm1d, torch.nn.BatchNorm2d)):
        module.eval()
        for param in module.parameters():
            param.requires_grad = False
            
for param in model.decoder_with_head.parameters():
    param.requires_grad = True

In [300]:
## Train / Val Split
from sklearn.model_selection import train_test_split

file_indices = list(range(len(all_file_data)))
train_file_indices, val_file_indices = train_test_split(
    file_indices, 
    test_size=0.2, 
    random_state=42
)

context_length = 512  # Reduced from 512 for more windows
stride = 64  # Smaller stride for more overlap

train_windows, train_masks, train_sources = create_windows_from_files(
    train_file_indices, all_file_data, context_length, stride
)

val_windows, val_masks, val_sources = create_windows_from_files(
    val_file_indices, all_file_data, context_length, stride
)

print(f"Train windows: {len(train_windows)} from {len(train_file_indices)} files")
print(f"Val windows: {len(val_windows)} from {len(val_file_indices)} files")
print(f"Window shape: {train_windows[0].shape}")
print(f"Total windows: {len(train_windows) + len(val_windows)}")

train_dataset = [
    {
        'past_values': torch.FloatTensor(train_windows[i]),
        'past_observed_mask': torch.BoolTensor(train_masks[i])
    }
    for i in range(len(train_windows))
]

val_dataset = [
    {
        'past_values': torch.FloatTensor(val_windows[i]),
        'past_observed_mask': torch.BoolTensor(val_masks[i])
    }
    for i in range(len(val_windows))
]

train_file_set = set(train_file_indices)
val_file_set = set(val_file_indices)
assert len(train_file_set & val_file_set) == 0, "❌ Data leakage detected!"
print("✓ No data leakage - train and val files are completely separate")
print(f"Dataset ready: {len(train_dataset)} train samples, {len(val_dataset)} val samples")


Train windows: 383 from 65 files
Val windows: 82 from 17 files
Window shape: (512, 6)
Total windows: 465
✓ No data leakage - train and val files are completely separate
Dataset ready: 383 train samples, 82 val samples


## Evaluate Fine-tuned Model


In [301]:
batch_size = 16
learning_rate = 1e-4
num_epochs = 20

training_args = TrainingArguments(
    output_dir="./tspulse_coolant_finetuned",
    overwrite_output_dir=True,
    num_train_epochs=num_epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    learning_rate=learning_rate,
    warmup_ratio=0.1,
    weight_decay=0.01,
    dataloader_num_workers=0,
    report_to="none",
    save_total_limit=2,
)

# Only trainable parameters (decoder)
trainable_params = [p for p in model.parameters() if p.requires_grad]
print(f"Trainable parameters: {sum(p.numel() for p in trainable_params):,}")

total_steps = len(train_dataset) * num_epochs // (batch_size * 4)
print(f'{len(train_dataset)}, {num_epochs}, {batch_size}')
print(f"Total training steps: {total_steps}")

optimizer = AdamW(trainable_params, lr=learning_rate, weight_decay=0.01)

scheduler = OneCycleLR(
    optimizer,
    max_lr=learning_rate,
    total_steps=total_steps,
    pct_start=0.1
)

print(f"Total training steps: {total_steps}")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    optimizers=(optimizer, scheduler),
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=5,
            early_stopping_threshold=0.001
        )
    ]
)

print("✓ Trainer ready!")
trainer.train()

Trainable parameters: 286,272
383, 20, 16
Total training steps: 119
Total training steps: 119
✓ Trainer ready!


/Users/darenpalmer/Desktop/UCL/CS/fyp.nosync/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


RuntimeError: required rank 4 tensor to use channels_last format